<a href="https://colab.research.google.com/github/nmemranhussain/DiaVigil/blob/main/notebooks/02_diavigil_clinical_feature_engineering_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment & Partition Loading**

Mount Google Drive and load dev_cohort.parquet.

In [1]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

dev_df = pd.read_parquet(os.path.join(DATA_PROCESSED, 'dev_cohort.parquet'))
print(f"Loaded Development Cohort: {len(dev_df):,} encounters")
print(f"Features available: {dev_df.shape[1]}")

Mounted at /content/drive
Loaded Development Cohort: 80,239 encounters
Features available: 51


**Cell 2: Elixhauser Comorbidity Mapping Engine**

Map raw ICD-9 codes across diag_1, diag_2, and diag_3 into standardized clinical comorbidity categories based on Quan's adaptation of the Elixhauser Comorbidity Index.

In [2]:
import re

# Standardized Elixhauser ICD-9 mapping patterns
ELIXHAUSER_ICD9_RULES = {
    'elix_congestive_heart_failure': r'^(39891|40201|40211|40291|40401|40403|40411|40413|40491|40493|428)',
    'elix_cardiac_arrhythmias': r'^(42610|42611|42613|4262|4263|4264|42651|42652|42653|4266|4267|4268|4270|4272|4273|4276|4278|4279|7850|V450|V533)',
    'elix_valvular_disease': r'^(0932|394|395|396|397|424|7463|7464|7465|7466|V422|V433)',
    'elix_pulmonary_circulation': r'^(416|4179)',
    'elix_peripheral_vascular': r'^(440|441|442|443|444|4471|5571|5579|V434)',
    'elix_hypertension_uncomplicated': r'^(401)',
    'elix_hypertension_complicated': r'^(402|403|404|405)',
    'elix_diabetes_complicated': r'^(250[4-9])',
    'elix_renal_failure': r'^(585|586|V420|V451|V56)',
    'elix_fluid_electrolyte': r'^(276)',
    'elix_chronic_pulmonary': r'^(490|491|492|493|494|495|496|500|501|502|503|504|505)',
    'elix_coagulopathy': r'^(286|2871|2873|2874|2875)'
}

def extract_elixhauser_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Vectorized comorbidity mapping across primary, secondary, and tertiary diagnoses."""
    diag_cols = ['diag_1', 'diag_2', 'diag_3']
    comorb_features = pd.DataFrame(index=df.index)

    # Clean diagnostic string columns
    cleaned_diags = df[diag_cols].fillna('').astype(str).apply(lambda col: col.str.strip().str.replace('.', '', regex=False))

    for flag_name, pattern in ELIXHAUSER_ICD9_RULES.items():
        compiled_regex = re.compile(pattern)
        # Match pattern against any of the 3 diagnosis fields
        matches = cleaned_diags.apply(
            lambda row: any(bool(compiled_regex.match(val)) for val in row if val),
            axis=1
        )
        comorb_features[flag_name] = matches.astype(int)

    return comorb_features

# Validate comorbidity extraction
sample_comorbs = extract_elixhauser_flags(dev_df.head(100))
print("Comorbidity mapping verified. Extracted categories:")
print(sample_comorbs.columns.tolist())

Comorbidity mapping verified. Extracted categories:
['elix_congestive_heart_failure', 'elix_cardiac_arrhythmias', 'elix_valvular_disease', 'elix_pulmonary_circulation', 'elix_peripheral_vascular', 'elix_hypertension_uncomplicated', 'elix_hypertension_complicated', 'elix_diabetes_complicated', 'elix_renal_failure', 'elix_fluid_electrolyte', 'elix_chronic_pulmonary', 'elix_coagulopathy']


**Cell 3: Custom Scikit-Learn Clinical Feature Transformer**

Encapsulate all clinical transformations (Elixhauser mapping, pharmacotherapy complexity, healthcare utilization index, glycemic risk indicators) inside a Scikit-Learn transformer so transformations can be fitted strictly on training folds during cross-validation.

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin

class ClinicalDomainFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.medication_columns = [
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
            'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
            'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
            'metformin-pioglitazone'
        ]

    def fit(self, X, y=None):
        # Stateless transformer: no parameters to estimate from y
        return self

    def transform(self, X):
        X_out = X.copy()

        # 1. Elixhauser Comorbidity Features
        comorb_df = extract_elixhauser_flags(X_out)
        for col in comorb_df.columns:
            X_out[col] = comorb_df[col].values

        # Comorbidity Burden: Total count of active chronic conditions
        X_out['total_elixhauser_burden'] = comorb_df.sum(axis=1).values

        # 2. Healthcare Utilization Index (Weighted prior encounters)
        X_out['healthcare_utilization_index'] = (
            X_out['number_outpatient'].astype(float) +
            (2.0 * X_out['number_emergency'].astype(float)) +
            (3.0 * X_out['number_inpatient'].astype(float))
        )

        # 3. Pharmacotherapy Regimen Complexity
        active_meds = X_out[self.medication_columns].apply(lambda row: (row != 'No').sum(), axis=1)
        med_titrations = X_out[self.medication_columns].apply(lambda row: row.isin(['Up', 'Down']).sum(), axis=1)

        X_out['active_diabetes_med_count'] = active_meds
        X_out['dosage_titration_count'] = med_titrations
        X_out['on_insulin'] = (X_out['insulin'] != 'No').astype(int)

        # 4. Clinical Encounter Intensity
        X_out['encounter_severity_index'] = (
            X_out['time_in_hospital'].astype(float) *
            (X_out['num_medications'].astype(float) / (X_out['num_lab_procedures'].astype(float) + 1.0))
        )

        # 5. Glycemic Monitoring Indicator
        # 0: None, 1: Norm, 2: >7 or >8 (Elevated/High Risk)
        X_out['glycemic_status_code'] = X_out['A1Cresult'].map({
            'None': 0,
            'Norm': 1,
            '>7': 2,
            '>8': 2
        }).fillna(0).astype(int)

        return X_out

# Test transformation
transformer = ClinicalDomainFeatureEngineer()
dev_engineered = transformer.transform(dev_df)
print(f"Features expanded from {dev_df.shape[1]} to {dev_engineered.shape[1]} clinical columns.")

Features expanded from 51 to 70 clinical columns.


**Cell 4: Complete Preprocessing Pipeline Construction**

Construct a leak-proof ColumnTransformer that applies targeted imputations, scalers, and categorical encodings to the engineered clinical feature set.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

# Define feature subsets
NUMERICAL_FEATURES = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'healthcare_utilization_index',
    'total_elixhauser_burden', 'active_diabetes_med_count',
    'dosage_titration_count', 'encounter_severity_index'
]

BINARY_COMORBIDITY_FEATURES = [col for col in dev_engineered.columns if col.startswith('elix_')] + ['on_insulin', 'glycemic_status_code']

CATEGORICAL_FEATURES = [
    'race', 'gender', 'age', 'admission_type_id',
    'discharge_disposition_id', 'admission_source_id', 'change', 'diabetesMed'
]

# Pipelines per data type
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Master Preprocessor
preprocessor = ColumnTransformer([
    ('num', num_pipeline, NUMERICAL_FEATURES),
    ('comorb', 'passthrough', BINARY_COMORBIDITY_FEATURES),
    ('cat', cat_pipeline, CATEGORICAL_FEATURES)
])

# Full Feature Engineering Pipeline
full_feature_pipeline = Pipeline([
    ('domain_engineering', ClinicalDomainFeatureEngineer()),
    ('preprocessor', preprocessor)
])

print("Clinical feature pipeline successfully assembled.")

Clinical feature pipeline successfully assembled.


**Cell 5: Serialization of Feature Pipeline & Feature Metadata**

Serialize the feature pipeline code and persist metadata definitions to disk. This ensures Phase 3 models can train and cross-validate without re-declaring feature transformers.

In [5]:
import joblib

pipeline_path = os.path.join(BASE_DIR, 'data/processed/feature_engineering_pipeline.joblib')
joblib.dump(full_feature_pipeline, pipeline_path)

# Verify transform output dimensions
X_sample = dev_df.drop(columns=['target', 'encounter_id', 'patient_nbr'])
X_transformed_sample = full_feature_pipeline.fit_transform(X_sample.head(200))

print(f"Pipeline persisted to: {pipeline_path}")
print(f"Transformed feature matrix shape: {X_transformed_sample.shape}")

Pipeline persisted to: /content/drive/MyDrive/DiaVigil/data/processed/feature_engineering_pipeline.joblib
Transformed feature matrix shape: (200, 64)
